# 9 · Feature-based EKF SLAM

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/09_ekf_slam.md)).
The state stacks the robot pose and every landmark, $y = [x;\ m_1; \dots; m_n]$, with one joint covariance. Prediction
moves only the robot: $\bar P_{xx} = FP_{xx}F^T + WQW^T$, $\bar P_{xm} = FP_{xm}$. A new landmark is appended with
$m = g(x, z)$ and covariance $G_xP_{xx}G_x^T + G_zRG_z^T$, **plus** its cross-covariance $G_yP$ with everything already
in the state. A reading of one landmark corrects all of them through the correlations. Landmark uncertainty never
grows, relative positions become exact, absolute accuracy is bounded by the initial robot uncertainty, and a loop
closure collapses the uncertainty of the whole loop. Over long runs EKF SLAM becomes over-confident.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## A loop around a ring of landmarks

Twenty-four landmarks on two rings, a sensor that sees 3 m ahead (180°), one loop of radius 6 m and a bit more.
Readings carry the simulator's landmark identifiers (known correspondences); JCBB is used at the end.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 13)[:-1]
landmarks = np.vstack([np.c_[8.0 * np.cos(angles), 8.0 * np.sin(angles)],
                       np.c_[4.0 * np.cos(angles + 0.26), 4.0 * np.sin(angles + 0.26)]])
dd, dt, k = se.DifferentialDrive(0.1, 0.5), 0.1, 2e-4
loop_steps = int(2 * np.pi * 6.0 / (0.5 * dt))
u = se.circle_controls(0.5, 6.0, int(loop_steps * 1.15))
sensor = se.RangeBearingSensor(max_range=3.0, fov=np.deg2rad(180), sigma_range=0.05, sigma_bearing=0.02)
x0 = np.array([6.0, 0.0, np.pi / 2])
run = se.simulate_landmark_run(x0, u, dt, dd, k, sensor, landmarks, 2, se.Rng(1))

slam = se.EkfSlam(x0, np.zeros((3, 3)))
sig, snapshots, lm_sigma = [0.0], {}, {}
for s, (dl, dr) in enumerate(run.wheel_travel):
    slam.predict_wheels(dd, dl, dr, k * abs(dl), k * abs(dr))
    slam.process_known(run.scans[s], sensor.R())
    sig.append(np.sqrt(np.trace(slam.pose_covariance[:2, :2])))
    for i in range(slam.n_landmarks):
        lm_sigma.setdefault(i, []).append((s, np.sqrt(np.trace(slam.landmark_covariance(i)))))
    if s + 1 in (loop_steps - 60, len(u)):
        snapshots[s + 1] = (slam.landmarks.copy(), [slam.landmark_covariance(i) for i in range(slam.n_landmarks)])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (kk, (L, C)) in zip(axes, snapshots.items()):
    plotting.new_axes(ax)
    ax.plot(landmarks[:, 0], landmarks[:, 1], "*", color="C1", ms=9, mec="k")
    plotting.plot_trajectory(ax, run.truth[: kk + 1], None, "k")
    for m_, c_ in zip(L, C):
        plotting.plot_ellipse(ax, m_, c_, 0.95, color="C0")
    ax.set_title(f"step {kk}: {'before' if kk < len(u) else 'after'} the loop closure (95 %)")
for i in (0, 5, 10, 20):
    if i in lm_sigma:
        t_, s_ = zip(*lm_sigma[i]); axes[2].plot(np.array(t_) * dt, s_, label=f"landmark {i}")
axes[2].plot(np.arange(len(sig)) * dt, sig, "k", lw=2, label="robot")
axes[2].set_xlabel("t [s]"); axes[2].set_ylabel("σ [m] (√trace)"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.show()

The correlation coefficients of the final covariance: after the loop closure every landmark is strongly correlated
with every other — the map is known much better in relative than in absolute terms.

In [ ]:
P = slam.covariance
d = np.sqrt(np.diag(P))
plt.imshow(P / np.outer(d, d), cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="correlation coefficient"); plt.title("robot (first 3) and landmark states"); plt.show()

### ✏️ Exercise 1 — state augmentation

Implement `augment(y, P, z, R)` (eq. 9.5–9.7) with `se.range_bearing_inverse` and its two Jacobians. Remember the
cross-covariance block $G_yP$, with $G_y = [G_x\; 0]$.

In [ ]:
def augment(y, P, z, R):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
s2 = se.EkfSlam(np.array([1.0, 2.0, 0.3]), np.diag([0.02, 0.03, 0.01]))
s2.add_landmark(np.array([2.0, 0.4]), sensor.R())
y_ref, P_ref = s2.state, s2.covariance
s2.add_landmark(np.array([3.0, -0.6]), sensor.R())
y_new, P_new = augment(y_ref, P_ref, np.array([3.0, -0.6]), sensor.R())
assert np.allclose(y_new, s2.state) and np.allclose(P_new, s2.covariance)
print("✔ augmentation matches, including the landmark–landmark cross-covariance:", P_new[3:5, 5:7].round(5).tolist())

### ✏️ Exercise 2 — the sparse prediction

Implement `predict(y, P, u, Q)` touching only the robot rows and columns (eq. 9.4), and check it against the dense form
$F_yPF_y^T + W_yQW_y^T$ and against the C++ filter.

In [ ]:
def predict(y, P, u, Q):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
u1, Q1 = np.array([0.3, 0.02, 0.1]), np.diag([1e-3, 1e-4, 1e-4])
y0, P0 = s2.state, s2.covariance
yp, Pp = predict(y0, P0, u1, Q1)
F = se.compose_jacobian_1(y0[:3], u1); W = se.compose_jacobian_2(y0[:3], u1)
Fy = np.eye(len(y0)); Fy[:3, :3] = F
Wy = np.zeros((len(y0), 3)); Wy[:3] = W
assert np.allclose(Pp, Fy @ P0 @ Fy.T + Wy @ Q1 @ Wy.T)
s2.predict_displacement(u1, Q1)
assert np.allclose(yp, s2.state) and np.allclose(Pp, s2.covariance)
print("✔ sparse prediction = dense prediction = C++")

### ✏️ Exercise 3 — landmark uncertainty never grows

Using `lm_sigma` from the run above is not enough (it is a trace). Re-run the filter and record, for every landmark,
the determinant of its covariance block after each step; return the largest relative increase observed. It must be
(numerically) zero.

In [ ]:
def largest_increase():
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
w_ = largest_increase()
print(f"largest relative increase of a landmark determinant: {w_:.2e}")
assert w_ < 1e-9

## SLAM without identifiers

The same run with unlabelled readings: JCBB associates each scan with the landmarks in the state, and an unpaired
reading becomes a landmark only if it passes the wide new-landmark gate (§9.5). Compare the size of the map with the
number of landmarks the sensor actually saw: any excess is a duplicated landmark, the typical failure of feature SLAM
without identifiers.

In [ ]:
f = se.EkfSlam(x0, np.zeros((3, 3)))
dropped = 0
for s, (dl, dr) in enumerate(run.wheel_travel):
    f.predict_wheels(dd, dl, dr, k * abs(dl), k * abs(dr))
    unlabelled = [se.Observation(-1, o.z) for o in run.scans[s]]
    r = f.process_jcbb(unlabelled, sensor.R())
    dropped += sum(j < 0 for j in r.landmark)
seen = {o.landmark for scan in run.scans for o in scan}
print(f"landmarks in the map: {f.n_landmarks}; distinct landmarks actually observed: {len(seen)}; "
      f"ambiguous readings dropped: {dropped}")
print(f"final pose error: JCBB {np.linalg.norm(f.pose[:2] - run.truth[-1, :2]):.3f} m, "
      f"known identifiers {np.linalg.norm(slam.pose[:2] - run.truth[-1, :2]):.3f} m")

## 🔨 Break it — forgetting the cross-covariance

Initialise each new landmark with the right mean and variance but **independent** of the robot and the map (the
common bug: `add_known_landmark` with the covariance of eq. 9.7's corner block only). The filter believes every
landmark is an independent source of information about the robot. Average robot NEES over 20 runs:

In [ ]:
def slam_nees(independent, seed, controls=u, k_enc=k):
    r = se.simulate_landmark_run(x0, controls, dt, dd, k_enc, sensor, landmarks, 2, se.Rng(seed))
    f = se.EkfSlam(x0, np.zeros((3, 3))); ids = {}; out = []
    for t, (dl, dr) in enumerate(r.wheel_travel):
        f.predict_wheels(dd, dl, dr, k_enc * abs(dl), k_enc * abs(dr))
        f.update([se.Observation(ids[o.landmark], o.z) for o in r.scans[t] if o.landmark in ids], sensor.R())
        for o in r.scans[t]:
            if o.landmark in ids:
                continue
            if independent:
                x = f.pose
                Gx = se.range_bearing_inverse_jacobian_pose(x, o.z); Gz = se.range_bearing_inverse_jacobian_measurement(x, o.z)
                ids[o.landmark] = f.add_known_landmark(se.range_bearing_inverse(x, o.z),
                                                       Gx @ f.pose_covariance @ Gx.T + Gz @ sensor.R() @ Gz.T)
            else:
                ids[o.landmark] = f.add_landmark(o.z, sensor.R())
        e = r.truth[t + 1] - f.pose; e[2] = se.wrap_angle(e[2])
        out.append(e @ np.linalg.solve(f.pose_covariance + 1e-12 * np.eye(3), e))
    return np.array(out)

lo, hi = se.average_chi2_bounds(3, 20)
for ind, name in ((False, "with cross-covariance"), (True, "independent landmarks")):
    N = np.mean([slam_nees(ind, sd) for sd in range(20)], axis=0)
    plt.semilogy(np.arange(len(N)) * dt, N, label=f"{name}: mean NEES {N.mean():.1f}")
plt.axhline(lo, color="k", ls="--"); plt.axhline(hi, color="k", ls="--")
plt.xlabel("t [s]"); plt.ylabel("average robot NEES (20 runs)"); plt.legend(); plt.show()

## 🔨 Break it — EKF SLAM over several loops

Drive the loop three times (8 runs each, to keep the notebook fast). With the encoder noise used so far the robot
NEES creeps upwards loop after loop; with ten times noisier encoders (larger heading uncertainty, worse
linearisation) the filter is far from consistent.

In [ ]:
u3 = np.vstack([u] * 3)
lo, hi = se.average_chi2_bounds(3, 8)
for k_enc in (2e-4, 2e-3):
    N = np.mean([slam_nees(False, sd, u3, k_enc) for sd in range(8)], axis=0)
    n1 = len(u)
    print(f"encoder noise {k_enc:g}: mean NEES loop 1 {N[:n1].mean():5.2f}, loop 2 {N[n1:2 * n1].mean():5.2f}, "
          f"loop 3 {N[2 * n1:].mean():5.2f}  (consistent: 3, 95 % band for one step [{lo:.2f}, {hi:.2f}])")

## What to remember

- EKF SLAM is EKF localisation on a growing state; prediction is $O(n)$, an update $O(n^2)$.
- New landmarks must carry their cross-covariance with the robot and the map; without it the filter is badly
  over-confident.
- Correlations make the map improve as a whole; a loop closure corrects everything at once.
- Absolute map accuracy is bounded by the initial robot uncertainty; relative accuracy keeps improving.
- Over long runs the linearised filter becomes inconsistent — the motivation for optimisation-based SLAM.